In [12]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import re

nltk.download('punkt')
nltk.download('stop_words')
nltk.download('wordnet')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\franc\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Error loading stop_words: Package 'stop_words' not found
[nltk_data]     in index
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\franc\AppData\Roaming\nltk_data...


True

In [3]:
df = pd.read_csv('judge-1377884607_tweet_product_company.csv',encoding='latin1')
df.head()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9093 entries, 0 to 9092
Data columns (total 3 columns):
 #   Column                                              Non-Null Count  Dtype 
---  ------                                              --------------  ----- 
 0   tweet_text                                          9092 non-null   object
 1   emotion_in_tweet_is_directed_at                     3291 non-null   object
 2   is_there_an_emotion_directed_at_a_brand_or_product  9093 non-null   object
dtypes: object(3)
memory usage: 213.2+ KB


In [5]:
df.describe()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
count,9092,3291,9093
unique,9065,9,4
top,RT @mention Marissa Mayer: Google Will Connect...,iPad,No emotion toward brand or product
freq,5,946,5389


In [6]:
df['is_there_an_emotion_directed_at_a_brand_or_product'].unique()

array(['Negative emotion', 'Positive emotion',
       'No emotion toward brand or product', "I can't tell"], dtype=object)

When cleaning the text I want to do the following:
1. Remove any starting or trailing white spaces
2. Ensure every word is lower case
3. Remove any numbers
4. Remove any special characters
5. Remove any emojis (if they dont count as special characters)
6. Tokenize
7. Lemmertize
8. Remove any stopwords
9. Remove any words that repeat consecutively
10. remove any letters that repeat more than twice
11. Remove any word that is connected to the # symbol (any words connected to special symbols)
12. Remove any links they all look like this {link}
13. Remove any empty lines.
14. Remove any . symbols that repeat multiple times
15. Remove special symbols connected to numbers 5,000
16. Remove @gmail.com or @yahoo.com
17. Remove .com

In [11]:
stop_words = set(stopwords.words('english'))
# print(len(stop_words))

In [13]:
import html
import emoji

def clean_text_custom(text):
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))

    # Unescape HTML characters like &amp;
    text = html.unescape(text)

    # Remove emojis
    text = emoji.replace_emoji(text, replace='')

    # Lowercase
    text = text.lower()

    # Remove retweet tag
    text = re.sub(r'^rt\s+', '', text)

    # Remove URLs
    text = re.sub(r'https?://\S+', '', text)

    # Remove hashtags, mentions, and cashtags
    text = re.sub(r'[@#\$]\w+', '', text)

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove repeated letters (3+)
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # Remove non-alphabet characters
    text = re.sub(r'[^a-z\s]', '', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenize
    tokens = word_tokenize(text)

    # Remove stopwords and short tokens
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]

    # Lemmatize
    lemmas = [lemmatizer.lemmatize(word) for word in tokens]

    return lemmas
